# Pengolahan Data Uji Coba Terbatas

Notebook ini disusun untuk mengolah data uji coba terbatas tahap 1 dan tahap 2 media web microlearning terintegrasi Teknik Feynman. Fokus pengolahan diarahkan pada kebutuhan penulisan karya ilmiah tingkat disertasi: statistik deskriptif, peningkatan pretest-posttest, respons pengguna, triangulasi temuan kualitatif, serta draft narasi pembahasan.


## Catatan Integritas Data

Sebelum membaca hasil, penting membedakan sumber data:

- Data `large_group_prepost.csv` diambil dari file field test seed sesuai catatan pada `CATATAN_SEED.md`.
- Data respons tahap 2 direkonstruksi dari ringkasan wawancara pada `DESC.md`.
- Data tahap 1 merupakan seed rekonstruksi untuk kebutuhan draf tabel dan eksplorasi analisis.

Karena itu, hasil notebook ini paling tepat dipakai sebagai perangkat penyusunan dan simulasi pembahasan. Jika akan dimasukkan sebagai bukti empiris final, narasi metodologis perlu menyebut status data secara transparan.


In [ ]:
from pathlib import Path
import csv
import math
import statistics as stats
import textwrap

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'uji_coba':
    BASE_DIR = BASE_DIR / 'data' / 'uji_coba'

OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

FILES = {
    'small_prepost': BASE_DIR / 'small_group_prepost.csv',
    'small_respons': BASE_DIR / 'small_group_respons.csv',
    'large_prepost': BASE_DIR / 'large_group_prepost.csv',
    'large_respons': BASE_DIR / 'large_group_respons.csv',
    'desc': BASE_DIR / 'DESC.md',
    'catatan': BASE_DIR / 'CATATAN_SEED.md',
}

PRE_ASPECTS = ['pre_pengorganisasian', 'pre_kejelasan', 'pre_ketepatan', 'pre_strategi', 'pre_metakognitif']
POST_ASPECTS = ['post_pengorganisasian', 'post_kejelasan', 'post_ketepatan', 'post_strategi', 'post_metakognitif']
ASPECT_NAMES = ['pengorganisasian', 'kejelasan', 'ketepatan', 'strategi', 'metakognitif']
RESP_ASPECTS = ['tampilan', 'navigasi', 'multimedia', 'interaktivitas', 'materi', 'teknik_feynman']

print('Base:', BASE_DIR)
print('Output:', OUTPUT_DIR)
for key, path in FILES.items():
    print(f'{key:14s}:', 'OK' if path.exists() else 'MISSING', path)


In [ ]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    return rows

def write_csv(path, rows, fieldnames=None):
    if not fieldnames:
        fieldnames = list(rows[0].keys()) if rows else []
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def fnum(value):
    if value is None or value == '':
        return None
    return float(value)

def mean(values):
    vals = [fnum(v) for v in values if fnum(v) is not None]
    return stats.mean(vals) if vals else None

def sd(values):
    vals = [fnum(v) for v in values if fnum(v) is not None]
    return stats.stdev(vals) if len(vals) > 1 else 0.0

def median(values):
    vals = [fnum(v) for v in values if fnum(v) is not None]
    return stats.median(vals) if vals else None

def fmt(value, digits=2):
    if value is None:
        return '-'
    return f'{value:.{digits}f}'

def category_percent(value):
    if value >= 85:
        return 'Sangat baik'
    if value >= 70:
        return 'Baik'
    if value >= 55:
        return 'Cukup'
    return 'Kurang'

def n_gain(pre, post):
    if pre is None or post is None or pre >= 100:
        return None
    return (post - pre) / (100 - pre)

def category_ngain(value):
    if value is None:
        return '-'
    if value >= 0.70:
        return 'Tinggi'
    if value >= 0.30:
        return 'Sedang'
    return 'Rendah'

def cohens_d_paired(pre_values, post_values):
    diffs = [post - pre for pre, post in zip(pre_values, post_values)]
    if len(diffs) < 2:
        return None
    s = stats.stdev(diffs)
    return None if s == 0 else stats.mean(diffs) / s

def cohens_d_independent(a, b):
    if len(a) < 2 or len(b) < 2:
        return None
    s1, s2 = stats.stdev(a), stats.stdev(b)
    pooled = math.sqrt(((len(a)-1)*s1*s1 + (len(b)-1)*s2*s2) / (len(a)+len(b)-2))
    return None if pooled == 0 else (stats.mean(a) - stats.mean(b)) / pooled

def ci95_mean(values):
    vals = [fnum(v) for v in values if fnum(v) is not None]
    if len(vals) < 2:
        return (None, None)
    se = stats.stdev(vals) / math.sqrt(len(vals))
    m = stats.mean(vals)
    return (m - 1.96 * se, m + 1.96 * se)

def markdown_table(rows, columns=None):
    if not rows:
        return ''
    columns = columns or list(rows[0].keys())
    header = '| ' + ' | '.join(columns) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = []
    for row in rows:
        body.append('| ' + ' | '.join(str(row.get(col, '')) for col in columns) + ' |')
    return '\n'.join([header, sep] + body)

def show_table(title, rows, columns=None, limit=None):
    print('\n' + title)
    data = rows[:limit] if limit else rows
    print(markdown_table(data, columns))

small_pre = read_csv(FILES['small_prepost'])
small_resp = read_csv(FILES['small_respons'])
large_pre = read_csv(FILES['large_prepost'])
large_resp = read_csv(FILES['large_respons'])
desc_text = FILES['desc'].read_text(encoding='utf-8') if FILES['desc'].exists() else ''

print('Loaded rows:', {
    'small_prepost': len(small_pre),
    'small_respons': len(small_resp),
    'large_prepost': len(large_pre),
    'large_respons': len(large_resp),
})


In [ ]:
def validate_rows(rows, expected_width=None):
    issues = []
    for i, row in enumerate(rows, start=1):
        blanks = [k for k, v in row.items() if v == '']
        if blanks:
            issues.append({'row': i, 'issue': 'blank_cells', 'columns': ', '.join(blanks)})
    return issues

def validate_respons(rows):
    issues = []
    for row in rows:
        total = sum(int(float(row[k])) for k in RESP_ASPECTS)
        avg = round(total / len(RESP_ASPECTS), 2)
        pct = round(total / (len(RESP_ASPECTS) * 4) * 100, 2)
        if total != int(float(row['total'])):
            issues.append({'id': row['id'], 'issue': 'total_mismatch', 'expected': total, 'found': row['total']})
        if abs(avg - float(row['rata_rata'])) > 0.01:
            issues.append({'id': row['id'], 'issue': 'average_mismatch', 'expected': avg, 'found': row['rata_rata']})
        if abs(pct - float(row['persentase'])) > 0.01:
            issues.append({'id': row['id'], 'issue': 'percent_mismatch', 'expected': pct, 'found': row['persentase']})
    return issues

def validate_prepost(rows):
    issues = []
    for row in rows:
        pre_sum = sum(int(float(row[k])) for k in PRE_ASPECTS)
        post_sum = sum(int(float(row[k])) for k in POST_ASPECTS)
        pre_pct = round(pre_sum / 75 * 100, 2)
        post_pct = round(post_sum / 75 * 100, 2)
        if abs(pre_pct - float(row['pre_total'])) > 0.05:
            issues.append({'id': row['id'], 'issue': 'pre_total_mismatch', 'expected': pre_pct, 'found': row['pre_total']})
        if abs(post_pct - float(row['post_total'])) > 0.05:
            issues.append({'id': row['id'], 'issue': 'post_total_mismatch', 'expected': post_pct, 'found': row['post_total']})
    return issues

validation = []
for name, rows in [('small_prepost', small_pre), ('small_respons', small_resp), ('large_prepost', large_pre), ('large_respons', large_resp)]:
    validation.extend({'file': name, **issue} for issue in validate_rows(rows))
validation.extend({'file': 'small_prepost', **issue} for issue in validate_prepost(small_pre))
validation.extend({'file': 'large_prepost', **issue} for issue in validate_prepost(large_pre))
validation.extend({'file': 'small_respons', **issue} for issue in validate_respons(small_resp))
validation.extend({'file': 'large_respons', **issue} for issue in validate_respons(large_resp))

if validation:
    show_table('Temuan validasi data', validation)
else:
    print('Validasi data: tidak ditemukan sel kosong atau ketidaksesuaian rumus total.')


In [ ]:
def enrich_prepost(rows, stage):
    enriched = []
    for row in rows:
        r = dict(row)
        pre = fnum(r['pre_total'])
        post = fnum(r['post_total'])
        gain = post - pre
        ng = n_gain(pre, post)
        r.update({
            'tahap': stage,
            'gain_total': round(gain, 2),
            'n_gain': round(ng, 3) if ng is not None else '',
            'kategori_n_gain': category_ngain(ng),
            'kategori_pre': category_percent(pre),
            'kategori_post': category_percent(post),
        })
        for pre_col, post_col, aspect in zip(PRE_ASPECTS, POST_ASPECTS, ASPECT_NAMES):
            r[f'gain_{aspect}'] = int(float(r[post_col])) - int(float(r[pre_col]))
        enriched.append(r)
    return enriched

def enrich_respons(rows, stage):
    enriched = []
    for row in rows:
        r = dict(row)
        pct = fnum(r['persentase'])
        r.update({'tahap': stage, 'kategori': category_percent(pct)})
        enriched.append(r)
    return enriched

small_pre_e = enrich_prepost(small_pre, 'Tahap 1')
large_pre_e = enrich_prepost(large_pre, 'Tahap 2')
small_resp_e = enrich_respons(small_resp, 'Tahap 1')
large_resp_e = enrich_respons(large_resp, 'Tahap 2')

show_table('Contoh data pretest-posttest tahap 2 setelah enrichment', large_pre_e, ['id', 'nama', 'kelompok', 'pre_total', 'post_total', 'gain_total', 'n_gain', 'kategori_n_gain'], limit=10)
show_table('Contoh data respons tahap 2 setelah enrichment', large_resp_e, ['id', 'nama', 'kelompok', 'persentase', 'kategori'], limit=10)


In [ ]:
def prepost_summary(rows, label, group_key=None):
    groups = {}
    if group_key:
        for row in rows:
            groups.setdefault(row[group_key], []).append(row)
    else:
        groups[label] = rows
    result = []
    for group, data in groups.items():
        pre = [fnum(r['pre_total']) for r in data]
        post = [fnum(r['post_total']) for r in data]
        gains = [fnum(r['gain_total']) for r in data]
        ngains = [fnum(r['n_gain']) for r in data]
        lo, hi = ci95_mean(gains)
        result.append({
            'tahap/kelompok': group,
            'n': len(data),
            'mean_pre': fmt(mean(pre)),
            'sd_pre': fmt(sd(pre)),
            'mean_post': fmt(mean(post)),
            'sd_post': fmt(sd(post)),
            'mean_gain': fmt(mean(gains)),
            'sd_gain': fmt(sd(gains)),
            'ci95_gain': f'{fmt(lo)} - {fmt(hi)}',
            'mean_n_gain': fmt(mean(ngains), 3),
            'kategori_n_gain': category_ngain(mean(ngains)),
            'd_paired': fmt(cohens_d_paired(pre, post)),
        })
    return result

summary_stage = prepost_summary(small_pre_e, 'Tahap 1') + prepost_summary(large_pre_e, 'Tahap 2')
summary_group_stage2 = prepost_summary(large_pre_e, 'Tahap 2', group_key='kelompok')

write_csv(OUTPUT_DIR / 'summary_prepost_by_stage.csv', summary_stage)
write_csv(OUTPUT_DIR / 'summary_stage2_by_group.csv', summary_group_stage2)

show_table('Ringkasan pretest-posttest per tahap', summary_stage)
show_table('Ringkasan pretest-posttest tahap 2 per kelompok', summary_group_stage2)

exp_gain = [fnum(r['gain_total']) for r in large_pre_e if r.get('kelompok') == 'eksperimen']
ctl_gain = [fnum(r['gain_total']) for r in large_pre_e if r.get('kelompok') == 'kontrol']
print('\nEffect size selisih gain eksperimen-kontrol tahap 2 (Cohen d independent):', fmt(cohens_d_independent(exp_gain, ctl_gain)))


In [ ]:
aspect_rows = []
for aspect in ASPECT_NAMES:
    gains = [fnum(r[f'gain_{aspect}']) for r in large_pre_e]
    exp = [fnum(r[f'gain_{aspect}']) for r in large_pre_e if r.get('kelompok') == 'eksperimen']
    ctl = [fnum(r[f'gain_{aspect}']) for r in large_pre_e if r.get('kelompok') == 'kontrol']
    aspect_rows.append({
        'aspek': aspect,
        'mean_gain_total': fmt(mean(gains)),
        'mean_gain_eksperimen': fmt(mean(exp)),
        'mean_gain_kontrol': fmt(mean(ctl)),
        'sd_gain_total': fmt(sd(gains)),
    })

write_csv(OUTPUT_DIR / 'aspect_gain_stage2.csv', aspect_rows)
show_table('Gain per aspek keterampilan berbicara pada tahap 2', aspect_rows)


In [ ]:
def response_summary(rows, label, group_key=None):
    groups = {}
    if group_key:
        for row in rows:
            groups.setdefault(row[group_key], []).append(row)
    else:
        groups[label] = rows
    result = []
    for group, data in groups.items():
        pcts = [fnum(r['persentase']) for r in data]
        result.append({
            'tahap/kelompok': group,
            'n': len(data),
            'mean_persentase': fmt(mean(pcts)),
            'sd': fmt(sd(pcts)),
            'median': fmt(median(pcts)),
            'min': fmt(min(pcts)),
            'max': fmt(max(pcts)),
            'kategori': category_percent(mean(pcts)),
        })
    return result

resp_stage = response_summary(small_resp_e, 'Tahap 1') + response_summary(large_resp_e, 'Tahap 2')
resp_group_stage2 = response_summary(large_resp_e, 'Tahap 2', group_key='kelompok')

show_table('Ringkasan respons pengguna per tahap', resp_stage)
show_table('Ringkasan respons pengguna tahap 2 per kelompok', resp_group_stage2)

resp_aspect_rows = []
for aspect in RESP_ASPECTS:
    vals = [fnum(r[aspect]) for r in large_resp_e]
    pct = mean(vals) / 4 * 100
    resp_aspect_rows.append({
        'aspek': aspect,
        'mean_skor': fmt(mean(vals)),
        'persentase': fmt(pct),
        'kategori': category_percent(pct),
    })
write_csv(OUTPUT_DIR / 'response_aspect_summary_stage2.csv', resp_aspect_rows)
show_table('Ringkasan respons tahap 2 per aspek', resp_aspect_rows)


In [ ]:
qualitative_codes = [
    {'tahap': 'Tahap 1', 'tema': 'Kendala autentikasi', 'indikasi': '4 dari 6 responden', 'makna': 'Login menjadi hambatan utama dan perlu diperbaiki sebelum skenario lebih luas.'},
    {'tahap': 'Tahap 1', 'tema': 'Beban video/server', 'indikasi': 'Keluhan video berat dan harus berpindah ke YouTube', 'makna': 'Media perlu menyajikan video secara lebih ringan dan terintegrasi.'},
    {'tahap': 'Tahap 1', 'tema': 'Kedalaman aktivitas Feynman', 'indikasi': 'Konten dinilai masih seperti kuis biasa', 'makna': 'Aktivitas perlu menuntun peserta menjelaskan kembali, bukan hanya menjawab kuis.'},
    {'tahap': 'Tahap 2', 'tema': 'Keterlaksanaan alur', 'indikasi': '7 dari 10 lancar; 3 dari 10 mengalami kendala', 'makna': 'Alur pembelajaran sudah relatif stabil tetapi masih perlu perbaikan navigasi dan login.'},
    {'tahap': 'Tahap 2', 'tema': 'Navigasi mobile', 'indikasi': '2 dari 10 mengalami kendala navigasi', 'makna': 'Antarmuka perlu dioptimalkan untuk pola akses mahasiswa yang dominan mobile.'},
    {'tahap': 'Tahap 2', 'tema': 'Login/lupa password', 'indikasi': '1 dari 10 mengalami kendala login', 'makna': 'Perlu fallback autentikasi agar aktivitas tidak berhenti karena masalah akun.'},
    {'tahap': 'Tahap 2', 'tema': 'Level gating', 'indikasi': 'Mayoritas menyarankan pembatas untuk naik level', 'makna': 'Progression rule dapat meningkatkan tantangan dan menjaga keterurutan belajar.'},
    {'tahap': 'Tahap 2', 'tema': 'Sistem poin', 'indikasi': 'Mahasiswa sangat menyukai poin ketika memberi feedback', 'makna': 'Gamifikasi ringan mendukung keterlibatan dan motivasi partisipatif.'},
    {'tahap': 'Tahap 2', 'tema': 'Video microlearning', 'indikasi': 'Sekitar 85% sangat baik; sisanya biasa saja', 'makna': 'Video ringkas diterima baik, tetapi variasi kualitas pengalaman tetap perlu dicermati.'},
    {'tahap': 'Tahap 2', 'tema': 'Kode masuk case sensitive', 'indikasi': 'Banyak kendala masuk pada fitur uji', 'makna': 'Validasi input perlu dibuat lebih toleran atau instruksi teknis dibuat lebih eksplisit.'},
    {'tahap': 'Tahap 2', 'tema': 'Observasi gestur saat tes', 'indikasi': 'Saran promotor', 'makna': 'Modul pre/post perlu memfasilitasi observasi gestur sebagai bagian konstruk keterampilan berbicara.'},
    {'tahap': 'Tahap 2', 'tema': 'Log analitik', 'indikasi': 'Saran kopromotor', 'makna': 'Data log dapat menjadi temuan penggunaan dan memperkuat evaluasi proses.'},
]

write_csv(OUTPUT_DIR / 'qualitative_codes.csv', qualitative_codes)
show_table('Matriks kode kualitatif dari DESC.md', qualitative_codes)


In [ ]:
def svg_bar_chart(path, title, items, width=900, height=None, bar_height=28):
    height = height or (80 + len(items) * 46)
    label_width = 220
    chart_width = width - label_width - 80
    max_val = max(v for _, v in items) if items else 1
    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="24" y="32" font-family="Arial" font-size="20" font-weight="700" fill="#111827">{title}</text>'
    ]
    y = 62
    for label, value in items:
        bar_w = 0 if max_val == 0 else value / max_val * chart_width
        lines.append(f'<text x="24" y="{y + 20}" font-family="Arial" font-size="13" fill="#374151">{label}</text>')
        lines.append(f'<rect x="{label_width}" y="{y}" width="{bar_w:.1f}" height="{bar_height}" rx="3" fill="#2563eb"/>')
        lines.append(f'<text x="{label_width + bar_w + 8:.1f}" y="{y + 19}" font-family="Arial" font-size="13" fill="#111827">{value:.2f}</text>')
        y += 46
    lines.append('</svg>')
    path.write_text('\n'.join(lines), encoding='utf-8')

stage2_group_items = []
for row in summary_group_stage2:
    stage2_group_items.append((f"{row['tahap/kelompok']} pre", float(row['mean_pre'])))
    stage2_group_items.append((f"{row['tahap/kelompok']} post", float(row['mean_post'])))
svg_bar_chart(OUTPUT_DIR / 'stage2_prepost_by_group.svg', 'Rata-rata Pretest dan Posttest Tahap 2', stage2_group_items)

aspect_items = [(row['aspek'], float(row['mean_gain_total'])) for row in aspect_rows]
svg_bar_chart(OUTPUT_DIR / 'stage2_gain_by_aspect.svg', 'Rata-rata Gain Aspek Keterampilan Tahap 2', aspect_items)

resp_items = [(row['aspek'], float(row['persentase'])) for row in resp_aspect_rows]
svg_bar_chart(OUTPUT_DIR / 'stage2_response_aspect.svg', 'Persentase Respons per Aspek Tahap 2', resp_items)

print('SVG charts written to:', OUTPUT_DIR)


In [ ]:
# Optional inferential statistics. SciPy is not required; if available, this cell adds p-values.
try:
    from scipy import stats as scipy_stats
except Exception:
    scipy_stats = None

inferential_rows = []
for label, rows in [('Tahap 1', small_pre_e), ('Tahap 2', large_pre_e)]:
    pre = [fnum(r['pre_total']) for r in rows]
    post = [fnum(r['post_total']) for r in rows]
    if scipy_stats:
        t_res = scipy_stats.ttest_rel(post, pre)
        try:
            w_res = scipy_stats.wilcoxon([post[i] - pre[i] for i in range(len(pre))])
            wilcoxon_p = w_res.pvalue
        except Exception:
            wilcoxon_p = None
        inferential_rows.append({
            'tahap': label,
            'paired_t': fmt(t_res.statistic, 4),
            'p_t': fmt(t_res.pvalue, 4),
            'p_wilcoxon': fmt(wilcoxon_p, 4) if wilcoxon_p is not None else '-',
        })
    else:
        inferential_rows.append({
            'tahap': label,
            'paired_t': 'SciPy tidak tersedia',
            'p_t': '-',
            'p_wilcoxon': '-',
        })

write_csv(OUTPUT_DIR / 'inferential_optional.csv', inferential_rows)
show_table('Uji inferensial opsional', inferential_rows)


In [ ]:
def get_summary(rows, key):
    for row in rows:
        if row['tahap/kelompok'] == key:
            return row
    return {}

s1 = get_summary(summary_stage, 'Tahap 1')
s2 = get_summary(summary_stage, 'Tahap 2')
exp = get_summary(summary_group_stage2, 'eksperimen')
ctl = get_summary(summary_group_stage2, 'kontrol')
r1 = response_summary(small_resp_e, 'Tahap 1')[0]
r2 = response_summary(large_resp_e, 'Tahap 2')[0]

discussion = f"""# Draft Pembahasan Uji Coba Terbatas

## Uji Coba Terbatas Tahap 1

Uji coba terbatas tahap 1 dilakukan kepada 6 mahasiswa untuk memperoleh gambaran awal mengenai kelayakan media, kemudahan alur penggunaan, dan respons mahasiswa terhadap media web microlearning terintegrasi Teknik Feynman. Berdasarkan hasil pengolahan data, rerata respons mahasiswa pada tahap ini sebesar {r1.get('mean_persentase')}% dan termasuk kategori {r1.get('kategori')}. Hasil pretest-posttest juga menunjukkan adanya peningkatan, yaitu dari rerata {s1.get('mean_pre')} pada pretest menjadi {s1.get('mean_post')} pada posttest.

Berdasarkan hasil tersebut, media pada tahap awal dapat dinyatakan sudah dapat digunakan, tetapi masih memerlukan revisi. Catatan mahasiswa menunjukkan bahwa kendala utama terdapat pada login, video yang terasa berat ketika digunakan bersamaan, serta aktivitas yang masih terasa seperti kuis biasa dan belum sepenuhnya mengarahkan mahasiswa untuk menjelaskan kembali materi. Oleh karena itu, revisi setelah tahap 1 difokuskan pada perbaikan bug login, penyediaan alternatif autentikasi, penyematan video agar tidak perlu berpindah ke YouTube, serta penambahan ragam evaluasi dan latihan rekam untuk memperkuat penerapan Teknik Feynman.

## Uji Coba Terbatas Tahap 2

Setelah produk direvisi, uji coba terbatas tahap 2 dilakukan kepada 10 mahasiswa, terdiri atas 5 mahasiswa dari kelompok eksperimen dan 5 mahasiswa dari kelompok kontrol. Pada tahap ini, rerata respons mahasiswa meningkat menjadi {r2.get('mean_persentase')}% dan termasuk kategori {r2.get('kategori')}. Hasil pretest-posttest menunjukkan peningkatan dari rerata {s2.get('mean_pre')} menjadi {s2.get('mean_post')}. Jika dilihat berdasarkan kelompok, mahasiswa eksperimen meningkat dari {exp.get('mean_pre')} menjadi {exp.get('mean_post')}, sedangkan mahasiswa kontrol meningkat dari {ctl.get('mean_pre')} menjadi {ctl.get('mean_post')}.

Berdasarkan tabel dan hasil pengolahan tersebut, media web microlearning terintegrasi Teknik Feynman pada tahap 2 dinyatakan lebih praktis dan lebih siap digunakan dibandingkan tahap sebelumnya. Simulasi pembelajaran secara umum berjalan lancar; 7 dari 10 mahasiswa dapat mengikuti alur tanpa kendala berarti, sedangkan 3 mahasiswa mengalami kendala berupa 2 kendala navigasi dan 1 kendala login karena lupa password. Mahasiswa juga memberikan respons positif terhadap video microlearning dan sistem poin, tetapi menyarankan adanya pembatas atau syarat untuk naik level agar proses belajar terasa lebih menantang.

Masukan pada tahap 2 digunakan sebagai dasar revisi lanjutan. Revisi diarahkan pada penyederhanaan navigasi agar lebih ramah perangkat mobile, penambahan kuis atau syarat antarbagian, perbaikan alur modul uji, serta penyediaan alternatif pengumpulan link apabila perekaman langsung mengalami kendala. Selain itu, fitur log penggunaan perlu dioptimalkan agar waktu tinggal, progres mahasiswa, dan hambatan teknis dapat diamati sebagai data pendukung evaluasi media.

## Simpulan Uji Coba Terbatas

Secara keseluruhan, hasil uji coba terbatas menunjukkan bahwa media mengalami peningkatan kualitas dari tahap 1 ke tahap 2. Pada tahap 1, media sudah menunjukkan potensi tetapi masih menghadapi kendala teknis dan pedagogis. Setelah revisi, tahap 2 menunjukkan respons mahasiswa yang sangat baik, peningkatan hasil pretest-posttest, serta alur pembelajaran yang lebih lancar. Dengan demikian, media web microlearning terintegrasi Teknik Feynman dinyatakan layak untuk dilanjutkan ke tahap pengujian berikutnya dengan beberapa perbaikan pada navigasi, mekanisme naik level, modul uji, dan fitur log analitik.

> Catatan: bagian ini disusun mengikuti pola pembahasan ringkas seperti referensi, yaitu menekankan hasil tabel/gambar, kategori kelayakan, peningkatan pretest-posttest, dan revisi produk. Detail statistik tambahan tetap tersedia di notebook sebagai data pendukung, tetapi tidak perlu seluruhnya masuk ke narasi utama BAB IV.
""".strip()

(OUTPUT_DIR / 'draft_pembahasan_uji_coba.md').write_text(discussion + '\n', encoding='utf-8')
print(discussion)


In [ ]:
print('Output files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)


## Snapshot Hasil Utama

Sel ini adalah snapshot statis dari hasil terakhir saat notebook dibuat. Jika data CSV berubah, jalankan ulang sel kode agar output dan pembahasan ikut diperbarui.

### Ringkasan Pretest-Posttest per Tahap

| tahap/kelompok | n | mean_pre | sd_pre | mean_post | sd_post | mean_gain | sd_gain | ci95_gain | mean_n_gain | kategori_n_gain | d_paired |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Tahap 1 | 6 | 62.22 | 4.01 | 69.32 | 4.62 | 7.10 | 0.70 | 6.54 - 7.66 | 0.191 | Rendah | 10.16 |
| Tahap 2 | 10 | 71.60 | 4.74 | 87.47 | 2.89 | 15.87 | 5.53 | 12.44 - 19.29 | 0.543 | Sedang | 2.87 |

### Ringkasan Tahap 2 per Kelompok

| tahap/kelompok | n | mean_pre | sd_pre | mean_post | sd_post | mean_gain | sd_gain | ci95_gain | mean_n_gain | kategori_n_gain | d_paired |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| eksperimen | 5 | 72.54 | 6.83 | 89.86 | 1.53 | 17.32 | 7.83 | 10.46 - 24.18 | 0.594 | Sedang | 2.21 |
| kontrol | 5 | 70.67 | 1.34 | 85.08 | 1.48 | 14.41 | 1.47 | 13.12 - 15.70 | 0.491 | Sedang | 9.80 |

### Respons Tahap 2 per Aspek

| aspek | mean_skor | persentase | kategori |
| --- | --- | --- | --- |
| tampilan | 4.00 | 100.00 | Sangat baik |
| navigasi | 3.70 | 92.50 | Sangat baik |
| multimedia | 3.80 | 95.00 | Sangat baik |
| interaktivitas | 4.00 | 100.00 | Sangat baik |
| materi | 4.00 | 100.00 | Sangat baik |
| teknik_feynman | 3.60 | 90.00 | Sangat baik |


# Draft Pembahasan Uji Coba Terbatas

## Uji Coba Terbatas Tahap 1

Uji coba terbatas tahap 1 dilakukan kepada 6 mahasiswa untuk memperoleh gambaran awal mengenai kelayakan media, kemudahan alur penggunaan, dan respons mahasiswa terhadap media web microlearning terintegrasi Teknik Feynman. Berdasarkan hasil pengolahan data, rerata respons mahasiswa pada tahap ini sebesar 65.97% dan termasuk kategori Cukup. Hasil pretest-posttest juga menunjukkan adanya peningkatan, yaitu dari rerata 62.22 pada pretest menjadi 69.32 pada posttest.

Berdasarkan hasil tersebut, media pada tahap awal dapat dinyatakan sudah dapat digunakan, tetapi masih memerlukan revisi. Catatan mahasiswa menunjukkan bahwa kendala utama terdapat pada login, video yang terasa berat ketika digunakan bersamaan, serta aktivitas yang masih terasa seperti kuis biasa dan belum sepenuhnya mengarahkan mahasiswa untuk menjelaskan kembali materi. Oleh karena itu, revisi setelah tahap 1 difokuskan pada perbaikan bug login, penyediaan alternatif autentikasi, penyematan video agar tidak perlu berpindah ke YouTube, serta penambahan ragam evaluasi dan latihan rekam untuk memperkuat penerapan Teknik Feynman.

## Uji Coba Terbatas Tahap 2

Setelah produk direvisi, uji coba terbatas tahap 2 dilakukan kepada 10 mahasiswa, terdiri atas 5 mahasiswa dari kelompok eksperimen dan 5 mahasiswa dari kelompok kontrol. Pada tahap ini, rerata respons mahasiswa meningkat menjadi 96.25% dan termasuk kategori Sangat baik. Hasil pretest-posttest menunjukkan peningkatan dari rerata 71.60 menjadi 87.47. Jika dilihat berdasarkan kelompok, mahasiswa eksperimen meningkat dari 72.54 menjadi 89.86, sedangkan mahasiswa kontrol meningkat dari 70.67 menjadi 85.08.

Berdasarkan tabel dan hasil pengolahan tersebut, media web microlearning terintegrasi Teknik Feynman pada tahap 2 dinyatakan lebih praktis dan lebih siap digunakan dibandingkan tahap sebelumnya. Simulasi pembelajaran secara umum berjalan lancar; 7 dari 10 mahasiswa dapat mengikuti alur tanpa kendala berarti, sedangkan 3 mahasiswa mengalami kendala berupa 2 kendala navigasi dan 1 kendala login karena lupa password. Mahasiswa juga memberikan respons positif terhadap video microlearning dan sistem poin, tetapi menyarankan adanya pembatas atau syarat untuk naik level agar proses belajar terasa lebih menantang.

Masukan pada tahap 2 digunakan sebagai dasar revisi lanjutan. Revisi diarahkan pada penyederhanaan navigasi agar lebih ramah perangkat mobile, penambahan kuis atau syarat antarbagian, perbaikan alur modul uji, serta penyediaan alternatif pengumpulan link apabila perekaman langsung mengalami kendala. Selain itu, fitur log penggunaan perlu dioptimalkan agar waktu tinggal, progres mahasiswa, dan hambatan teknis dapat diamati sebagai data pendukung evaluasi media.

## Simpulan Uji Coba Terbatas

Secara keseluruhan, hasil uji coba terbatas menunjukkan bahwa media mengalami peningkatan kualitas dari tahap 1 ke tahap 2. Pada tahap 1, media sudah menunjukkan potensi tetapi masih menghadapi kendala teknis dan pedagogis. Setelah revisi, tahap 2 menunjukkan respons mahasiswa yang sangat baik, peningkatan hasil pretest-posttest, serta alur pembelajaran yang lebih lancar. Dengan demikian, media web microlearning terintegrasi Teknik Feynman dinyatakan layak untuk dilanjutkan ke tahap pengujian berikutnya dengan beberapa perbaikan pada navigasi, mekanisme naik level, modul uji, dan fitur log analitik.

> Catatan: bagian ini disusun mengikuti pola pembahasan ringkas seperti referensi, yaitu menekankan hasil tabel/gambar, kategori kelayakan, peningkatan pretest-posttest, dan revisi produk. Detail statistik tambahan tetap tersedia di notebook sebagai data pendukung, tetapi tidak perlu seluruhnya masuk ke narasi utama BAB IV.
